# HydrAI — Physics-Informed Neural Network (PINN) with PFR ODE Residuals

Purpose: train a Physics-Informed Neural Network (PINNPFR) on PFR simulation data.
The model is penalised not only for data mismatch but also for violating the
governing PFR equations, turning the surrogate from purely data-driven into
physics-informed.

Prerequisites: run Main_2 and Main_3 so `data/processed/features_targets_*.pkl` exists.

---

## Physics constraints

The composite loss is:
```
L_total = λ_data · MSE(ŷ, y) + λ_phys · L_physics
```
with a curriculum: data-only for `CURRICULUM_WARMUP_EPOCHS`, then physics added.

### Algebraic constraints (no Cantera; work with lumped species)
1. **Ideal-gas EOS**: `p − ρ R T / M = 0`  
2. **Mass conservation**: `ρ u A − ṁ = 0`  
3. **Species sum**: `Σ Y_lump_k = 1`  
4. **Species non-negativity**: `Y_lump_k ≥ 0`  

### ODE constraints (autograd on z/L)
5. **Energy ODE**: `dT/d(z/L) = L π D q / (ṁ cp)` — wall-heat simplified form;  
   enforced via `torch.autograd.grad` on the `relative_position` input.

### Cantera ODE residuals (optional, `USE_CANTERA_RESIDUALS=True`)
6. Full species ODE: `dY_k/dz = W_k ω_k / (ρ u)` — requires raw (unlumped) species.  
   Gated behind `USE_CANTERA_RESIDUALS`; disabled by default for lumped datasets.

## Architecture
Same `SimpleNN` as Main_7 (3 hidden layers + dropout). Physics is enforced
externally in the training loop — no architectural changes needed.

## Collocation strategy
For each mini-batch, a set of **collocation points** is sampled:
inlet conditions drawn from training data, z/L replaced with uniform random
samples in [0, 1]. Physics residuals are evaluated at these novel z positions
without requiring labels.

## Overfitting controls
Same as Main_7: dropout, run-level train/test split, scalers fit on train only,
shuffled minibatches, `ReduceLROnPlateau` on test R², early stopping, best-
checkpoint restore. Physics loss is an additional inductive bias, not a
regulariser per se — a growing data/physics gap signals balance problems.


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 1. SETUP & IMPORTS
# ════════════════════════════════════════════════════════════════════════════
import os
import sys
import glob
import json
import re
import joblib
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Project root
current_dir = Path(os.getcwd())
project_root = current_dir if (current_dir / "src").exists() else current_dir.parent
sys.path.insert(0, str(project_root))
os.chdir(project_root)

from src.utils.plot_style import setup_matplotlib
from src.utils.run_log import start_run_log
from src.ml.dataframe_pickle import load_portable_pickle
from src.physics import compute_algebraic_residuals, build_colloc_input
from src.physics.pfr_residuals import (
    inv_scale_y, inv_scale_X,
    compute_energy_ode_residual,
    R_UNIVERSAL,
)
from src.utils.training_progress_log import (
    MAIN_8_STEM,
    training_progress_log_path,
    append_training_progress,
    init_training_progress_log,
)

setup_matplotlib()
start_run_log('Main_8_PINN_PFR')
print("Libraries imported successfully.")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 1b. DEVICE (CPU / NVIDIA CUDA / Apple MPS)
# ════════════════════════════════════════════════════════════════════════════
def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

device = get_device()
print(f"PyTorch version: {torch.__version__}")
print(f"Using device:    {device}")
if device.type == "cuda":
    print(f"GPU name:        {torch.cuda.get_device_name(0)}")
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 2. PATHS & FLAGS
# ════════════════════════════════════════════════════════════════════════════

# I/O flags
IF_PLOT_SHOWN  = True
IF_PLOT_EXPORT = True
IF_MODEL_EXPORT = True

# Training progress log (same pattern as Main_6 / Main_7)
WRITE_TRAINING_PROGRESS_LOG = True
TRAINING_PROGRESS_LOG = training_progress_log_path(project_root, MAIN_8_STEM)

# Smoke-test / memory guard: None = full dataset
FULL_PROFILE_MAX_ROWS = None

# Physics loss flag: set True if training on raw (unlumped) species data
# and the mechanism file is available for Cantera ODE residuals.
# Default False: algebraic + energy-ODE constraints only.
USE_CANTERA_RESIDUALS = False

# Paths
CONFIG_PATH        = project_root / "configs" / "ml" / "ml_training_config.json"
PROCESSED_DATA_DIR = project_root / "data" / "processed"
EXPORT_DIR         = project_root / "models"
FIG_DIR = project_root / "outputs" / "figures" / "Main_8_PINN_PFR"

print(f"Plot: {IF_PLOT_SHOWN}  |  Export: {IF_MODEL_EXPORT}")
print(f"USE_CANTERA_RESIDUALS: {USE_CANTERA_RESIDUALS}")
print(f"FIG_DIR: {FIG_DIR}")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 3. LOAD CONFIG & DATA
# ════════════════════════════════════════════════════════════════════════════
# Keys consumed from ml_training_config.json:
#   neural_network.*         — architecture and training (same as Main_7)
#   pinn.loss_weights.*      — per-term physics loss multipliers
#   pinn.training.*          — curriculum schedule and collocation budget

if CONFIG_PATH.exists():
    with open(CONFIG_PATH) as f:
        config = json.load(f)
else:
    config = {}
    print(f"[WARN] Config not found: {CONFIG_PATH}. Using defaults.")

TEST_SIZE    = config.get("test_size", 0.2)
RANDOM_STATE = config.get("random_state", 42)

NN_CONFIG     = config.get("neural_network", {})
EPOCHS        = int(NN_CONFIG.get("epochs", 400))
BATCH_SIZE    = int(NN_CONFIG.get("batch_size", 256))
LEARNING_RATE = float(NN_CONFIG.get("learning_rate", 1e-3))
H1            = int(NN_CONFIG.get("h1", 128))
H2            = int(NN_CONFIG.get("h2", 64))
H3            = int(NN_CONFIG.get("h3", 32))
DROPOUT       = float(NN_CONFIG.get("dropout", 0.1))

PINN_CONFIG   = config.get("pinn", {})
LW            = PINN_CONFIG.get("loss_weights", PINN_CONFIG)   # fallback: flat keys
TR            = PINN_CONFIG.get("training",     PINN_CONFIG)   # fallback: flat keys

LAMBDA_DATA           = float(LW.get("lambda_data",         1.0))
LAMBDA_PHYS           = float(LW.get("lambda_phys",         0.1))
LAMBDA_EOS            = float(LW.get("lambda_eos",          1.0))
LAMBDA_MASS           = float(LW.get("lambda_mass",         1.0))
LAMBDA_SPECIES_SUM    = float(LW.get("lambda_species_sum",  1.0))
LAMBDA_SPECIES_NONNEG = float(LW.get("lambda_species_nonneg", 0.5))
LAMBDA_ENERGY_ODE     = float(LW.get("lambda_energy_ode",   1.0))

CURRICULUM_WARMUP  = int(TR.get("curriculum_warmup_epochs", 20))
N_COLLOC_PER_BATCH = int(TR.get("n_colloc_per_batch",      256))
PHYS_LOSS_FREQ     = int(TR.get("phys_loss_freq",            1))
USE_CANTERA_RESIDUALS = bool(TR.get("use_cantera_residuals", False))

# ── Load processed features/targets pickle ───────────────────────────────────
pkl_files = sorted(PROCESSED_DATA_DIR.glob("features_targets_*.pkl"))
if not pkl_files:
    raise FileNotFoundError(
        f"No features_targets_*.pkl found in {PROCESSED_DATA_DIR}. "
        "Run Main_2 and Main_3 first."
    )
pkl_path = pkl_files[-1]
print(f"Loading: {pkl_path.name}")

data_dict = load_portable_pickle(pkl_path)
df_features = data_dict["df_features"]
df_target   = data_dict["df_target"]

if FULL_PROFILE_MAX_ROWS is not None:
    df_features = df_features.iloc[:FULL_PROFILE_MAX_ROWS].copy()
    df_target   = df_target.iloc[:FULL_PROFILE_MAX_ROWS].copy()

print(f"Features: {df_features.shape}  |  Targets: {df_target.shape}")
print(f"Config: epochs={EPOCHS}, batch={BATCH_SIZE}, lr={LEARNING_RATE}, "
      f"h={H1}/{H2}/{H3}, dropout={DROPOUT}")
print(f"Physics weights: λ_data={LAMBDA_DATA}, λ_phys={LAMBDA_PHYS}, "
      f"λ_eos={LAMBDA_EOS}, λ_mass={LAMBDA_MASS}, λ_energy={LAMBDA_ENERGY_ODE}")
print(f"Training schedule: warmup={CURRICULUM_WARMUP}, N_colloc={N_COLLOC_PER_BATCH}")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 4. FEATURES & TARGETS  (FULL AXIAL PROFILE; includes relative_position)
# ════════════════════════════════════════════════════════════════════════════
# Same task as Main_7: full axial profile with relative_position as a feature.
# Run-level train/test split (§5) prevents profile leakage.

feature_cols = [
    "initial_temperature_K", "initial_pressure_Pa",
    "reactor_length_m",      "reactor_diameter_m",
    "mass_flow_rate_kgps",   "heat_flux_Wm2",
    "z_position_m",          "relative_position",
]
if "reactant_type" in df_features.columns:
    feature_cols.append("reactant_type")
feature_cols = [c for c in feature_cols if c in df_features.columns]

if "relative_position" not in feature_cols:
    raise KeyError(
        "PINN requires 'relative_position' in df_features. "
        "Re-run Main_3 preprocessing if missing."
    )

# Run-identifying columns (exclude axial coordinates)
run_cols = [c for c in feature_cols if c not in ("z_position_m", "relative_position")]

# Column indices needed by physics residuals
Z_L_IDX  = feature_cols.index("relative_position")
Z_POS_IDX = feature_cols.index("z_position_m") if "z_position_m" in feature_cols else None
L_IDX    = feature_cols.index("reactor_length_m")
D_IDX    = feature_cols.index("reactor_diameter_m")
Q_IDX    = feature_cols.index("heat_flux_Wm2")
MDOT_IDX = feature_cols.index("mass_flow_rate_kgps")

feat_idx = {
    "reactor_length_m":    L_IDX,
    "reactor_diameter_m":  D_IDX,
    "heat_flux_Wm2":       Q_IDX,
    "mass_flow_rate_kgps": MDOT_IDX,
}

# Targets
primary_targets = [
    "temperature_K", "pressure_Pa", "velocity_ms", "density_kgm3",
    "mean_molecular_weight_kgkmol",
    "heat_capacity_cp_JkgK", "heat_capacity_cv_JkgK",
    "enthalpy_Jkg", "thermal_conductivity_WmK",
]
state_target_cols = [c for c in primary_targets if c in df_target.columns]

_lump_cols = [c for c in df_target.columns if c.startswith("Y_lump_")]
species_cols = sorted(_lump_cols) if _lump_cols else [c for c in df_target.columns if c.startswith("Y_")]

target_cols = state_target_cols + species_cols

# Require key physics targets
for req in ("temperature_K", "pressure_Pa", "density_kgm3", "velocity_ms",
             "mean_molecular_weight_kgkmol", "heat_capacity_cp_JkgK"):
    if req not in target_cols:
        print(f"[WARN] Physics target '{req}' not found — EOS / mass residuals may be skipped.")

# Target column indices for physics residuals
tgt_idx = {name: target_cols.index(name) for name in target_cols
           if name in (
               "temperature_K", "pressure_Pa", "velocity_ms", "density_kgm3",
               "mean_molecular_weight_kgkmol", "heat_capacity_cp_JkgK",
               "heat_capacity_cv_JkgK", "enthalpy_Jkg",
           )}
species_indices = [target_cols.index(c) for c in species_cols if c in target_cols]

# Build feature/target DataFrames
X_full = df_features[feature_cols].copy()
y_full = df_target[target_cols].copy()

# Label-encode reactant_type if present
label_encoder = LabelEncoder()
if "reactant_type" in feature_cols:
    X_full["reactant_type"] = label_encoder.fit_transform(X_full["reactant_type"])

# Drop rows with NaN
valid = ~(X_full.isna().any(axis=1) | y_full.isna().any(axis=1))
X_full = X_full[valid].reset_index(drop=True)
y_full = y_full[valid].reset_index(drop=True)

print(f"Feature cols ({len(feature_cols)}): {feature_cols}")
print(f"State targets ({len(state_target_cols)}): {state_target_cols}")
print(f"Species targets ({len(species_cols)}): {len(species_cols)} cols")
print(f"Total targets: {len(target_cols)}  |  Valid rows: {len(X_full):,}")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 5. TRAIN/TEST SPLIT & SCALING  (run-level split — no profile leakage)
# ════════════════════════════════════════════════════════════════════════════
# Run-level split: all axial rows from the same simulation run stay together.
# Same approach as Main_7 to ensure a fair comparison.

# Identify unique runs by their inlet fingerprint
run_id_col = "_run_id"
X_full[run_id_col] = X_full[run_cols].apply(
    lambda r: "|".join(str(round(v, 8)) for v in r), axis=1
)
unique_runs = X_full[run_id_col].unique()

rng = np.random.default_rng(RANDOM_STATE)
perm = rng.permutation(len(unique_runs))
n_test_runs = max(1, int(round(TEST_SIZE * len(unique_runs))))
test_runs  = set(unique_runs[perm[:n_test_runs]])
train_runs = set(unique_runs[perm[n_test_runs:]])

train_mask = X_full[run_id_col].isin(train_runs)
test_mask  = X_full[run_id_col].isin(test_runs)

X_train_df = X_full.loc[train_mask, feature_cols]
X_test_df  = X_full.loc[test_mask,  feature_cols]
y_train_df = y_full.loc[train_mask]
y_test_df  = y_full.loc[test_mask]

print(f"Train runs: {len(train_runs):,}  |  Test runs: {len(test_runs):,}")
print(f"Train rows: {len(X_train_df):,}  |  Test rows: {len(X_test_df):,}")

# ── Scalers (fit on training rows only) ───────────────────────────────────────
scaler_X = StandardScaler()
X_train_s = scaler_X.fit_transform(X_train_df)
X_test_s  = scaler_X.transform(X_test_df)

SCALE_TARGETS = True
if SCALE_TARGETS:
    scaler_y  = StandardScaler()
    y_train_s = scaler_y.fit_transform(y_train_df)
    y_test_s  = scaler_y.transform(y_test_df)
else:
    scaler_y  = None
    y_train_s = y_train_df.values
    y_test_s  = y_test_df.values

# Scaler stats as tensors for differentiable physics residuals
X_mean_t = torch.tensor(scaler_X.mean_,    dtype=torch.float32, device=device)
X_std_t  = torch.tensor(scaler_X.scale_,   dtype=torch.float32, device=device)
if scaler_y is not None:
    y_mean_t = torch.tensor(scaler_y.mean_,  dtype=torch.float32, device=device)
    y_std_t  = torch.tensor(scaler_y.scale_, dtype=torch.float32, device=device)
else:
    y_mean_t = torch.zeros(len(target_cols), device=device)
    y_std_t  = torch.ones(len(target_cols),  device=device)

print(f"X shape: {X_train_s.shape}  |  y shape: {y_train_s.shape}  |  scale_y={SCALE_TARGETS}")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 6. PINN ARCHITECTURE
# ════════════════════════════════════════════════════════════════════════════
# Dedicated PINNPFR class (src/models/pinn.py) — separate from the data-only
# SimpleNN used in Main_6 / Main_7 so that architectural changes here don't
# affect the pure-data surrogates.
# Physics residuals are enforced externally via torch.autograd.grad.

from src.models import PINNPFR

torch.manual_seed(RANDOM_STATE)

n_inputs  = X_train_s.shape[1]
n_outputs = y_train_s.shape[1]
model = PINNPFR(n_inputs, H1, H2, H3, n_outputs, dropout=DROPOUT).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(model)
print(f"\nTrainable parameters: {n_params:,}")
print(f"Inputs: {n_inputs}  (includes relative_position for PDE gradient)")
print(f"Outputs: {n_outputs}  ({len(state_target_cols)} state + {len(species_cols)} species)")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 7. COLLOCATION SAMPLING
# ════════════════════════════════════════════════════════════════════════════
# Collocation points = inlet conditions from training data + random z/L.
# Physics residuals are evaluated here without requiring labels.
#
# At each training step we:
#  1. Sample N_COLLOC inlet conditions uniformly from X_train_s
#  2. Assign uniform random z/L in [0, 1] to each
#  3. Construct the scaled input tensor via build_colloc_input
#     (differentiable in z_L_phys so autograd can trace through it)

# Inlet-only part of training features (no z columns — will be replaced)
_non_z_mask = torch.ones(n_inputs, dtype=torch.bool)
_non_z_mask[Z_L_IDX] = False
if Z_POS_IDX is not None:
    _non_z_mask[Z_POS_IDX] = False

X_train_t_full = torch.tensor(X_train_s, dtype=torch.float32, device=device)


def sample_colloc_batch(n_colloc: int, rng_state=None):
    """Return (X_colloc_s, z_L_phys) for a physics loss step.

    X_colloc_s : (n_colloc, n_feat) scaled input with random z/L embedded.
                 z/L column derived from z_L_phys (grad-enabled).
    z_L_phys   : (n_colloc, 1) physical z/L in [0, 1], requires_grad=True.
    """
    idx = torch.randint(0, X_train_t_full.shape[0], (n_colloc,))
    X_base = X_train_t_full[idx].detach()      # scaled inlet conditions

    z_L_phys = torch.rand(n_colloc, 1, device=device, dtype=torch.float32,
                          requires_grad=False)
    z_L_phys = z_L_phys.requires_grad_(True)   # enable grad after creation

    X_colloc = build_colloc_input(
        X_base, z_L_phys,
        feature_cols, X_mean_t, X_std_t,
    )
    return X_colloc, z_L_phys


# Quick sanity check
_Xc, _z = sample_colloc_batch(8)
assert _Xc.shape == (8, n_inputs), f"Colloc shape mismatch: {_Xc.shape}"
assert _z.requires_grad, "z_L_phys must have requires_grad=True"
print(f"Collocation batch shape: {tuple(_Xc.shape)}  |  z/L grad: {_z.requires_grad}")
del _Xc, _z


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 8. PHYSICS LOSS FUNCTION
# ════════════════════════════════════════════════════════════════════════════
# Combines algebraic constraints + energy ODE (autograd) at collocation points.
# The Cantera ODE path (USE_CANTERA_RESIDUALS) is implemented in
# src/physics/pfr_residuals.py but disabled by default for lumped species.

# ── Physics availability checks ──────────────────────────────────────────────
_phys_keys_present = all(k in tgt_idx for k in
    ("temperature_K", "pressure_Pa", "density_kgm3",
     "velocity_ms", "mean_molecular_weight_kgkmol", "heat_capacity_cp_JkgK"))
if not _phys_keys_present:
    print("[WARN] Some state targets missing — algebraic constraints partially disabled.")


def compute_physics_loss(
    model: nn.Module,
    n_colloc: int,
) -> dict:
    """Evaluate all physics residuals on a fresh collocation batch.

    Returns dict with keys:
      'algebraic' — sum of EOS + mass + species constraints
      'energy_ode' — energy ODE residual via autograd
      'total' — weighted sum
      'breakdown' — sub-residual scalars (detached) for logging
    """
    model.train()   # keep dropout active

    X_colloc, z_L_phys = sample_colloc_batch(n_colloc)

    # Forward pass — keep computation graph for autograd
    y_pred_s = model(X_colloc)

    # ── Inverse-transform to physical units (differentiable) ──────────────────
    y_pred_phys = inv_scale_y(y_pred_s, y_mean_t, y_std_t)
    X_phys      = inv_scale_X(X_colloc.detach(), X_mean_t, X_std_t)

    # ── 1. Algebraic constraints ──────────────────────────────────────────────
    alg_losses = {"eos": torch.tensor(0., device=device),
                  "mass": torch.tensor(0., device=device),
                  "species_sum": torch.tensor(0., device=device),
                  "species_nonneg": torch.tensor(0., device=device),
                  "total": torch.tensor(0., device=device)}

    if _phys_keys_present:
        alg_losses = compute_algebraic_residuals(
            y_pred_phys, X_phys,
            tgt_idx, feat_idx, species_indices,
            lambda_eos=LAMBDA_EOS,
            lambda_mass=LAMBDA_MASS,
            lambda_sum=LAMBDA_SPECIES_SUM,
            lambda_nonneg=LAMBDA_SPECIES_NONNEG,
        )

    # ── 2. Energy ODE residual via autograd ───────────────────────────────────
    energy_loss = torch.tensor(0., device=device)

    if "temperature_K" in tgt_idx and "heat_capacity_cp_JkgK" in tgt_idx:
        T_s = y_pred_s[:, tgt_idx["temperature_K"]]

        # d(T_scaled) / d(z_L_phys) — create_graph keeps second-order grads available
        dT_s_dz = torch.autograd.grad(
            T_s.sum(), z_L_phys,
            create_graph=True,
            retain_graph=True,
        )[0]   # shape (n_colloc, 1)

        # d(T_phys) / d(z/L) = dT_s_dz * std_T  (z_L_phys is physical so no std_z factor)
        std_T = y_std_t[tgt_idx["temperature_K"]]
        dT_phys_dz_L = dT_s_dz.squeeze(1) * std_T

        energy_loss = LAMBDA_ENERGY_ODE * compute_energy_ode_residual(
            dT_phys_dz_L, y_pred_phys, X_phys, tgt_idx, feat_idx,
        )

    total_phys = alg_losses["total"] + energy_loss

    return {
        "algebraic":   alg_losses["total"],
        "energy_ode":  energy_loss,
        "total":       total_phys,
        "breakdown":   {
            "eos":            alg_losses["eos"].item(),
            "mass":           alg_losses["mass"].item(),
            "species_sum":    alg_losses["species_sum"].item(),
            "species_nonneg": alg_losses["species_nonneg"].item(),
            "energy_ode":     energy_loss.item(),
        },
    }


print("Physics loss function defined.")
print(f"Algebraic constraints active: {_phys_keys_present}")
print(f"Energy ODE active: {'temperature_K' in tgt_idx and 'heat_capacity_cp_JkgK' in tgt_idx}")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 9. TENSORS & DATA LOADERS
# ════════════════════════════════════════════════════════════════════════════
X_train_t = torch.tensor(X_train_s, dtype=torch.float32, device=device)
X_test_t  = torch.tensor(X_test_s,  dtype=torch.float32, device=device)
y_train_t = torch.tensor(y_train_s, dtype=torch.float32, device=device)
y_test_t  = torch.tensor(y_test_s,  dtype=torch.float32, device=device)

train_ds     = TensorDataset(X_train_t, y_train_t)
batch_size   = min(BATCH_SIZE, len(train_ds))
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

# Update the full-train tensor used by collocation sampler
X_train_t_full = X_train_t

print(f"Tensors on device: {device}")
print(f"X_train_t: {tuple(X_train_t.shape)}  |  y_train_t: {tuple(y_train_t.shape)}")
print(f"Batches/epoch: {len(train_loader)}  |  batch_size: {batch_size}")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 10. TRAINING LOOP  (PINN with curriculum warmup)
# ════════════════════════════════════════════════════════════════════════════
# Phases:
#   Epochs 0 → CURRICULUM_WARMUP-1 : data loss only (warm-up)
#   Epochs CURRICULUM_WARMUP → END  : data loss + physics loss
#
# Same ReduceLROnPlateau + early-stop + best-checkpoint restore as Main_6/7.

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=4, threshold=1e-4, min_lr=1e-6
)

MIN_DELTA_R2              = 1e-4
EARLY_STOP_PATIENCE_EVALS = 12

train_loss_log      = []
epoch_log           = []
test_loss_log       = []
train_r2_log        = []
test_r2_log         = []
data_loss_log       = []   # data component only, at checkpoints
phys_loss_log       = []   # physics component only, at checkpoints
phys_breakdown_hist = []   # list of dicts, one per checkpoint

best_state_cpu = None
best_te_r2     = float("-inf")
best_epoch     = 0
stall_evals    = 0
EARLY_STOPPED  = False

if WRITE_TRAINING_PROGRESS_LOG:
    init_training_progress_log(TRAINING_PROGRESS_LOG)
    print(f"Training progress log: {TRAINING_PROGRESS_LOG}")

eval_every = max(1, EPOCHS // 40)
log_every  = max(1, EPOCHS // 20)


def eval_r2_phys_units(model, X_t, y_np, scaler_y):
    """Uniform-average R² in physical units on a full tensor (no grad)."""
    model.eval()
    with torch.no_grad():
        y_s = model(X_t).cpu().numpy()
    y_p = scaler_y.inverse_transform(y_s) if scaler_y is not None else y_s
    return float(r2_score(y_np, y_p, multioutput="uniform_average"))


print(f"Starting PINN training: {EPOCHS} epochs")
print(f"  Curriculum warmup: {CURRICULUM_WARMUP} epochs (data-only)")
print(f"  Physics λ_data={LAMBDA_DATA}, λ_phys={LAMBDA_PHYS}")
print(f"  Eval every {eval_every} epochs  |  log every {log_every} epochs")

y_train_np = y_train_df.values
y_test_np  = y_test_df.values

_phys_step = 0

for epoch in range(EPOCHS):
    model.train()
    epoch_data_loss = 0.0
    epoch_phys_loss = 0.0
    n_batches = 0

    physics_active = (epoch >= CURRICULUM_WARMUP)

    for step, (X_b, y_b) in enumerate(train_loader):
        optimizer.zero_grad()

        # ── Data loss ────────────────────────────────────────────────────────
        y_pred_s  = model(X_b)
        data_loss = criterion(y_pred_s, y_b)

        # ── Physics loss (after warmup, every PHYS_LOSS_FREQ steps) ──────────
        phys_loss_val = torch.tensor(0.0, device=device)

        if physics_active and (step % PHYS_LOSS_FREQ == 0):
            phys_out      = compute_physics_loss(model, N_COLLOC_PER_BATCH)
            phys_loss_val = phys_out["total"]

        # ── Total loss ───────────────────────────────────────────────────────
        total_loss = LAMBDA_DATA * data_loss + LAMBDA_PHYS * phys_loss_val
        total_loss.backward()
        optimizer.step()

        epoch_data_loss += data_loss.item()
        epoch_phys_loss += phys_loss_val.item()
        n_batches += 1

    avg_data = epoch_data_loss / max(n_batches, 1)
    avg_phys = epoch_phys_loss / max(n_batches, 1)
    train_loss_log.append(avg_data + LAMBDA_PHYS * avg_phys)

    # ── Periodic evaluation checkpoint ───────────────────────────────────────
    if epoch % eval_every == 0 or epoch == EPOCHS - 1:
        tr_r2 = eval_r2_phys_units(model, X_train_t, y_train_np, scaler_y)
        te_r2 = eval_r2_phys_units(model, X_test_t,  y_test_np,  scaler_y)

        model.eval()
        with torch.no_grad():
            te_mse = criterion(model(X_test_t), y_test_t).item()

        epoch_log.append(epoch)
        test_loss_log.append(te_mse)
        train_r2_log.append(tr_r2)
        test_r2_log.append(te_r2)
        data_loss_log.append(avg_data)
        phys_loss_log.append(avg_phys)

        if physics_active:
            model.train()
            phys_snapshot = compute_physics_loss(model, min(64, N_COLLOC_PER_BATCH))
            phys_breakdown_hist.append({"epoch": epoch, **phys_snapshot["breakdown"]})
            model.train()
        else:
            phys_breakdown_hist.append({"epoch": epoch,
                                         "eos": 0., "mass": 0.,
                                         "species_sum": 0., "species_nonneg": 0.,
                                         "energy_ode": 0.})

        scheduler.step(te_r2)

        if te_r2 > best_te_r2 + MIN_DELTA_R2:
            best_te_r2     = te_r2
            best_epoch     = epoch
            best_state_cpu = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            stall_evals    = 0
        else:
            stall_evals += 1

        if stall_evals >= EARLY_STOP_PATIENCE_EVALS:
            EARLY_STOPPED = True
            print(f"Early stop at epoch {epoch} (stall={stall_evals}, best R²={best_te_r2:.4f}@{best_epoch})")
            break

        if WRITE_TRAINING_PROGRESS_LOG:
            append_training_progress(
                TRAINING_PROGRESS_LOG, epoch, avg_data,
                test_mse=te_mse, train_r2=tr_r2, test_r2=te_r2,
                lr=float(optimizer.param_groups[0]["lr"]),
                is_checkpoint=True,
            )

    elif WRITE_TRAINING_PROGRESS_LOG and epoch % log_every == 0:
        append_training_progress(
            TRAINING_PROGRESS_LOG, epoch, avg_data,
            lr=float(optimizer.param_groups[0]["lr"]),
        )

    if epoch % (max(EPOCHS // 10, 1)) == 0:
        phys_flag = "[PHY]" if physics_active else "[WRM]"
        print(f"Epoch {epoch:4d}/{EPOCHS}  {phys_flag}  "
              f"data={avg_data:.4f}  phys={avg_phys:.5f}  "
              f"test_R²={test_r2_log[-1]:.4f}" if test_r2_log else
              f"Epoch {epoch:4d}/{EPOCHS}  {phys_flag}  data={avg_data:.4f}")

# ── Restore best checkpoint ───────────────────────────────────────────────────
if best_state_cpu is not None:
    model.load_state_dict({k: v.to(device) for k, v in best_state_cpu.items()})
    print(f"\nRestored best checkpoint: epoch {best_epoch}, test R²={best_te_r2:.4f}")

print(f"Training finished.  Early stopped: {EARLY_STOPPED}")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 10b. CONVERGENCE — DATA LOSS, PHYSICS LOSS, AND R²
# ════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# ── Panel A: total training loss vs epoch ────────────────────────────────────
ax = axes[0]
ax.plot(range(len(train_loss_log)), train_loss_log, color='b', lw=1.6, label='Train total (per epoch)')
ax.plot(epoch_log, test_loss_log, color='r', lw=1.6, label='Test MSE (checkpoints)')
ax.set_yscale('log')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE (standardised)')
ax.set_title('Convergence — total loss')
ax.grid(True, which='both', alpha=0.3)
ax.legend(fontsize=9)

# ── Panel B: data loss vs physics loss at checkpoints ────────────────────────
ax = axes[1]
ax.plot(epoch_log, data_loss_log, color='b', lw=1.6, label='Data loss (MSE scaled)')
ax.plot(epoch_log, phys_loss_log, color='m', lw=1.6, label='Physics loss (unweighted)')
if CURRICULUM_WARMUP < EPOCHS:
    ax.axvline(CURRICULUM_WARMUP, color='0.5', lw=1, ls='--', alpha=0.7,
               label=f'Curriculum end ({CURRICULUM_WARMUP})')
ax.set_yscale('log')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss component')
ax.set_title(f'Data vs physics loss  (λ_phys={LAMBDA_PHYS})')
ax.grid(True, which='both', alpha=0.3)
ax.legend(fontsize=9)

# ── Panel C: R² vs epoch ─────────────────────────────────────────────────────
ax = axes[2]
ax.plot(epoch_log, train_r2_log, color='b', lw=1.6, label='Train R²')
ax.plot(epoch_log, test_r2_log,  color='r', lw=1.6, label='Test R²')
ax.axhline(1.0, color='k', lw=0.8, alpha=0.4)
ax.axhline(0.0, color='k', lw=0.8, alpha=0.2)
if CURRICULUM_WARMUP < EPOCHS:
    ax.axvline(CURRICULUM_WARMUP, color='0.5', lw=1, ls='--', alpha=0.7)
ax.set_xlabel('Epoch')
ax.set_ylabel('R² (uniform average)')
ax.set_title('Convergence — R² (physical units)')
r2_min = min(min(train_r2_log, default=-0.1), min(test_r2_log, default=-0.1))
ax.set_ylim(bottom=min(-0.05, r2_min - 0.05), top=1.02)
ax.grid(True, alpha=0.3)
ax.legend(fontsize=9)

plt.suptitle('PINN Training Convergence', y=1.01)
plt.tight_layout()

if IF_PLOT_EXPORT:
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG_DIR / 'convergence_pinn.png', dpi=150, bbox_inches='tight')

if IF_PLOT_SHOWN:
    plt.show()
plt.close(fig)

# ── Physics breakdown convergence ─────────────────────────────────────────────
if phys_breakdown_hist:
    ep_bd  = [d["epoch"] for d in phys_breakdown_hist]
    keys   = ["eos", "mass", "species_sum", "species_nonneg", "energy_ode"]
    labels = ["EOS", "Mass conservation", "Species sum", "Species ≥ 0", "Energy ODE"]
    colors = ["b", "r", "m", "k", "c"]

    fig2, ax2 = plt.subplots(figsize=(11, 4))
    for k, lbl, col in zip(keys, labels, colors):
        vals = [d[k] for d in phys_breakdown_hist]
        if max(vals) > 0:
            ax2.plot(ep_bd, vals, lw=1.5, label=lbl, color=col)
    if CURRICULUM_WARMUP < EPOCHS:
        ax2.axvline(CURRICULUM_WARMUP, color='0.5', lw=1, ls='--', alpha=0.7,
                    label=f'Curriculum end ({CURRICULUM_WARMUP})')
    ax2.set_yscale('log')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Unweighted residual (log)')
    ax2.set_title('Physics constraint breakdown')
    ax2.grid(True, which='both', alpha=0.3)
    ax2.legend(fontsize=9, loc='upper right')
    plt.tight_layout()

    if IF_PLOT_EXPORT:
        fig2.savefig(FIG_DIR / 'convergence_physics_breakdown.png', dpi=150, bbox_inches='tight')
    if IF_PLOT_SHOWN:
        plt.show()
    plt.close(fig2)


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 11. TEST-SET EVALUATION
# ════════════════════════════════════════════════════════════════════════════
model.eval()
with torch.no_grad():
    y_pred_train_s = model(X_train_t).cpu().numpy()
    y_pred_test_s  = model(X_test_t).cpu().numpy()

if scaler_y is not None:
    y_pred_train = scaler_y.inverse_transform(y_pred_train_s)
    y_pred_test  = scaler_y.inverse_transform(y_pred_test_s)
else:
    y_pred_train = y_pred_train_s
    y_pred_test  = y_pred_test_s

train_r2  = r2_score(y_train_np, y_pred_train, multioutput='uniform_average')
test_r2   = r2_score(y_test_np,  y_pred_test,  multioutput='uniform_average')
test_mae  = mean_absolute_error(y_test_np,  y_pred_test,  multioutput='uniform_average')
test_rmse = np.sqrt(mean_squared_error(y_test_np, y_pred_test, multioutput='uniform_average'))

_ci = {n: i for i, n in enumerate(target_cols)}
_j_state   = [_ci[c] for c in state_target_cols   if c in _ci]
_j_species = [_ci[c] for c in species_cols         if c in _ci]

train_r2_state   = r2_score(y_train_np[:, _j_state],   y_pred_train[:, _j_state],   multioutput="uniform_average") if _j_state else float("nan")
test_r2_state    = r2_score(y_test_np[:, _j_state],    y_pred_test[:, _j_state],    multioutput="uniform_average") if _j_state else float("nan")
train_r2_species = r2_score(y_train_np[:, _j_species], y_pred_train[:, _j_species], multioutput="uniform_average") if _j_species else float("nan")
test_r2_species  = r2_score(y_test_np[:, _j_species],  y_pred_test[:, _j_species],  multioutput="uniform_average") if _j_species else float("nan")

# Per-target metrics
per_target_rows = []
for j, tgt in enumerate(target_cols):
    r2_j   = r2_score(y_test_np[:, j], y_pred_test[:, j])
    rmse_j = np.sqrt(mean_squared_error(y_test_np[:, j], y_pred_test[:, j]))
    mae_j  = mean_absolute_error(y_test_np[:, j],  y_pred_test[:, j])
    per_target_rows.append({"target": tgt, "R2": r2_j, "RMSE": rmse_j, "MAE": mae_j})
df_per_target = pd.DataFrame(per_target_rows)

print(f"\n{'─'*55}")
print(f"PINN TEST RESULTS")
print(f"{'─'*55}")
print(f"Test  R²  (all,     uniform):  {test_r2:.4f}")
print(f"Test  R²  (state):             {test_r2_state:.4f}")
print(f"Test  R²  (species):           {test_r2_species:.4f}")
print(f"Train R²  (all):               {train_r2:.4f}")
print(f"Train–Test gap:                {train_r2 - test_r2:.4f}")
print(f"Test  RMSE (avg):              {test_rmse:.6f}")
print(f"Test  MAE  (avg):              {test_mae:.6f}")
print(f"{'─'*55}")

print("\nTop-10 targets by test R²:")
print(df_per_target.sort_values('R2', ascending=False).head(10)[['target','R2','RMSE']].to_string(index=False))


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 11b. COMPARISON VS Main_6 / Main_7 BASELINES
# ════════════════════════════════════════════════════════════════════════════
# Load Main_6 and Main_7 manifests if they exist, then print a side-by-side
# comparison table. Falls back gracefully if manifests are missing.

def _load_manifest(stem):
    p = EXPORT_DIR / f"{stem}_manifest.json"
    if p.exists():
        with open(p) as f:
            return json.load(f)
    return None

m6 = _load_manifest("simple_nn_exit")
m7 = _load_manifest("simple_nn_full_profile")

print(f"{'Model':<30} {'Test R² (all)':>14} {'Test R² (state)':>16} {'Test R² (species)':>18} {'Train-Test gap':>15}")
print('─' * 80)

def _prow(name, m):
    if m is None:
        print(f"{name:<30} {'(manifest not found)':>14}")
        return
    met = m.get("metrics", {})
    r2  = met.get("test_r2", float("nan"))
    rs  = met.get("test_r2_state", float("nan"))
    rsp = met.get("test_r2_species", float("nan"))
    gap = met.get("train_r2", r2) - r2
    print(f"{name:<30} {r2:>14.4f} {rs:>16.4f} {rsp:>18.4f} {gap:>15.4f}")

_prow("Main_6 SimpleNN (exit only)",    m6)
_prow("Main_7 SimpleNN (full profile)", m7)
print(f"{'Main_8 PINN (full profile)':<30} {test_r2:>14.4f} {test_r2_state:>16.4f} "
      f"{test_r2_species:>18.4f} {train_r2 - test_r2:>15.4f}")
print('─' * 80)
print("Note: Main_6 is exit-plane only; Main_7 and Main_8 predict the full axial profile.")


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 12. ACTUAL VS PREDICTED — STATE & SPECIES (test set)
# ════════════════════════════════════════════════════════════════════════════
STATE_LABELS = {
    'temperature_K':                ('Temperature',           'K',        1.0),
    'pressure_Pa':                  ('Pressure',              'bar',      1e-5),
    'velocity_ms':                  ('Velocity',              'm/s',      1.0),
    'density_kgm3':                 ('Density',               'kg/m³',    1.0),
    'mean_molecular_weight_kgkmol': ('Mean MW',               'kg/kmol',  1.0),
    'heat_capacity_cp_JkgK':        ('Cp',                    'J/(kg·K)', 1.0),
    'heat_capacity_cv_JkgK':        ('Cv',                    'J/(kg·K)', 1.0),
    'enthalpy_Jkg':                 ('Enthalpy',              'J/kg',     1.0),
    'thermal_conductivity_WmK':     ('Thermal conductivity',  'W/(m·K)',  1.0),
}


def _label_triplet(tgt):
    if tgt in STATE_LABELS:
        return STATE_LABELS[tgt]
    short = tgt.replace("Y_lump_", "").replace("Y_", "")
    return (short, "", 1.0)


def _axis_title(tgt):
    name, unit, _ = _label_triplet(tgt)
    return f"{name} ({unit})" if unit else name


_state_set = set(state_target_cols)
all_scatter_targets = [c for c in (state_target_cols + species_cols) if c in target_cols]
n_cols   = 4
n_panels = len(all_scatter_targets)
n_rows   = (n_panels + n_cols - 1) // n_cols if n_panels else 0

PARITY_HEXBIN_MIN_POINTS = 500
use_hexbin = len(y_test_np) >= PARITY_HEXBIN_MIN_POINTS

if all_scatter_targets and n_rows > 0:
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.8 * n_cols, 3.0 * n_rows), squeeze=False)
    flat_axes = axes.ravel()

    for i, tgt in enumerate(all_scatter_targets):
        ax = flat_axes[i]
        tgt_idx_i = target_cols.index(tgt)
        _, unit, scale = _label_triplet(tgt)
        yt = y_test_np[:, tgt_idx_i] * scale
        yp = y_pred_test[:, tgt_idx_i] * scale
        r2_i = float(r2_score(yt, yp))
        lim = (min(yt.min(), yp.min()), max(yt.max(), yp.max()))
        ax.plot(lim, lim, 'r-', lw=2)
        if use_hexbin:
            ax.hexbin(yt, yp, gridsize=30, cmap='Blues', mincnt=1, linewidths=0.3)
        else:
            ax.scatter(yt, yp, s=10, alpha=0.25, edgecolors='none', c='b')
        ax.set_title(_axis_title(tgt), fontsize=8)
        ax.set_xlabel("Actual", fontsize=7)
        ax.set_ylabel("Predicted", fontsize=7)
        ax.tick_params(labelsize=7)
        ax.text(0.05, 0.93, f'R²={r2_i:.3f}', transform=ax.transAxes,
                fontsize=7, color='b', va='top')

    for j in range(len(all_scatter_targets), len(flat_axes)):
        flat_axes[j].set_visible(False)

    plt.suptitle('PINN — Actual vs Predicted (test set, 4-column)', y=1.005)
    plt.tight_layout()

    if IF_PLOT_EXPORT:
        FIG_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(FIG_DIR / 'actual_vs_predicted_scatter_by_target.png', dpi=130, bbox_inches='tight')
    if IF_PLOT_SHOWN:
        plt.show()
    plt.close(fig)


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 12b. RESIDUALS — STATE & SPECIES (test set)
# ════════════════════════════════════════════════════════════════════════════
if all_scatter_targets and n_rows > 0:
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.8 * n_cols, 3.0 * n_rows), squeeze=False)
    flat_axes = axes.ravel()

    for i, tgt in enumerate(all_scatter_targets):
        ax = flat_axes[i]
        tgt_idx_i = target_cols.index(tgt)
        _, unit, scale = _label_triplet(tgt)
        yt  = y_test_np[:, tgt_idx_i] * scale
        yp  = y_pred_test[:, tgt_idx_i] * scale
        res = yp - yt
        sigma = float(np.std(res))
        bias  = float(np.mean(res))

        ax.axhline(0, color='r', lw=1.4, zorder=3)
        ax.axhspan(-2*sigma, 2*sigma, color='0.85', alpha=0.5)
        ax.scatter(yt, res, s=10, alpha=0.25, edgecolors='none', c='b', zorder=4)
        ax.set_title(_axis_title(tgt), fontsize=8)
        ax.set_xlabel("Actual", fontsize=7)
        ax.set_ylabel("Residual", fontsize=7)
        ax.tick_params(labelsize=7)
        ax.text(0.05, 0.97, f'σ={sigma:.3g}\nbias={bias:.3g}',
                transform=ax.transAxes, fontsize=6.5, va='top')

    for j in range(len(all_scatter_targets), len(flat_axes)):
        flat_axes[j].set_visible(False)

    plt.suptitle('PINN — Residuals (test set)', y=1.005)
    plt.tight_layout()

    if IF_PLOT_EXPORT:
        fig.savefig(FIG_DIR / 'residuals_scatter_by_target.png', dpi=130, bbox_inches='tight')
    if IF_PLOT_SHOWN:
        plt.show()
    plt.close(fig)


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 12c. PER-TARGET R² BAR CHART
# ════════════════════════════════════════════════════════════════════════════
_state_set   = set(state_target_cols)
df_bar = df_per_target.copy()
df_bar['family'] = df_bar['target'].apply(lambda t: 'state' if t in _state_set else 'species')
df_bar = df_bar.sort_values('R2', ascending=False).reset_index(drop=True)

MAX_BARS    = 40
df_bar_plot = df_bar.head(MAX_BARS).iloc[::-1]
clipped_r2  = df_bar_plot['R2'].clip(lower=-0.05, upper=1.0)

fig_h = max(5.0, 0.28 * len(df_bar_plot))
fig, ax = plt.subplots(figsize=(9.5, fig_h))
ax.set_facecolor("0.94")

bars = ax.barh(df_bar_plot["target"], clipped_r2,
               color='white', edgecolor='0.35', linewidth=0.75)
for bar, fam in zip(bars, df_bar_plot["family"]):
    bar.set_hatch("" if fam == "state" else "///")

ax.axvline(0.9, color='lime', lw=1.0, ls='--', alpha=0.9, label='R² = 0.9 (good)')
ax.axvline(0.7, color='b',    lw=0.8, ls='--', alpha=0.7, label='R² = 0.7')
ax.axvline(0.0, color='r',    lw=1.0, ls='-',  alpha=0.6, label='R² = 0 (naïve baseline)')
ax.set_xlabel('Test R²')
ax.set_title('PINN — per-target test R² (state = solid, species = hatched)')

handles = [
    Patch(facecolor='white', edgecolor='0.35', hatch='',    label='State / thermo'),
    Patch(facecolor='white', edgecolor='0.35', hatch='///', label='Species / lumps'),
]
ax.legend(handles=handles, loc='lower right', fontsize=9)
ax.set_xlim(-0.07, 1.05)
ax.tick_params(axis='y', labelsize=8)
ax.grid(axis='x', alpha=0.4)
plt.tight_layout()

if IF_PLOT_EXPORT:
    fig.savefig(FIG_DIR / 'per_target_r2_bar.png', dpi=150, bbox_inches='tight')
if IF_PLOT_SHOWN:
    plt.show()
plt.close(fig)


In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 13. PHYSICS RESIDUAL MAGNITUDE — PINN vs UNTRAINED BASELINE
# ════════════════════════════════════════════════════════════════════════════
# Key PINN diagnostic: how much do physics residuals decrease compared to a
# random-weights network? A trained PINN should show substantially lower
# residuals, proving the physics constraints are actually enforced.
#
# Strategy:
#   1. Evaluate residuals on test samples using the trained PINN.
#   2. Evaluate the same residuals using an untrained (random-weights) PINNPFR
#      with identical architecture — this is the "no-physics" baseline.
#   3. Plot both profiles along z/L so the reduction is visually obvious.
#   4. Print a quantitative % reduction table (the key portfolio evidence).

N_DIAG_SAMPLES = min(2000, len(X_test_df))
_idx_diag      = np.random.default_rng(RANDOM_STATE + 99).choice(len(X_test_df), N_DIAG_SAMPLES, replace=False)

X_diag_s  = torch.tensor(X_test_s[_idx_diag],  dtype=torch.float32, device=device)
z_L_diag  = X_test_df.iloc[_idx_diag]["relative_position"].values

# Build collocation input (grad-enabled on z)
z_L_diag_t    = torch.tensor(z_L_diag[:, None], dtype=torch.float32, device=device, requires_grad=True)
X_diag_colloc = build_colloc_input(X_diag_s.detach(), z_L_diag_t, feature_cols, X_mean_t, X_std_t)


def _eval_residuals(net, X_colloc, z_t):
    """Return per-sample EOS, mass, and energy residual arrays for a given model."""
    net.eval()
    z_t2 = z_t.detach().requires_grad_(True)
    X2   = build_colloc_input(X_diag_s.detach(), z_t2, feature_cols, X_mean_t, X_std_t)
    y_s  = net(X2)
    y_ph = inv_scale_y(y_s, y_mean_t, y_std_t)
    X_ph = inv_scale_X(X_diag_s.detach(), X_mean_t, X_std_t)

    with torch.no_grad():
        T_p   = y_ph[:, tgt_idx["temperature_K"]]
        p_p   = y_ph[:, tgt_idx["pressure_Pa"]]
        rho_p = y_ph[:, tgt_idx["density_kgm3"]]
        M_p   = y_ph[:, tgt_idx["mean_molecular_weight_kgkmol"]].clamp(min=1.)
        u_p   = y_ph[:, tgt_idx["velocity_ms"]]
        D_p   = X_ph[:, feat_idx["reactor_diameter_m"]]
        m_p   = X_ph[:, feat_idx["mass_flow_rate_kgps"]]
        eos_per  = (((p_p - rho_p * R_UNIVERSAL * T_p / M_p) / p_p.abs().clamp(min=1e3)).pow(2)).cpu().numpy()
        A_p      = torch.pi * (D_p * 0.5).pow(2)
        mass_per = (((rho_p * u_p * A_p - m_p) / m_p.abs().clamp(min=1e-10)).pow(2)).cpu().numpy()

    # Energy ODE (needs grad)
    energy_per = np.zeros(N_DIAG_SAMPLES)
    if "temperature_K" in tgt_idx and "heat_capacity_cp_JkgK" in tgt_idx:
        y_s2  = net(X2)
        T_s2  = y_s2[:, tgt_idx["temperature_K"]]
        dT_dz = torch.autograd.grad(T_s2.sum(), z_t2, create_graph=False)[0]
        std_T = y_std_t[tgt_idx["temperature_K"]]
        dT_phys = dT_dz.squeeze(1) * std_T
        y_ph2 = inv_scale_y(y_s2, y_mean_t, y_std_t)
        cp    = y_ph2[:, tgt_idx["heat_capacity_cp_JkgK"]].clamp(min=100.)
        L_f   = X_ph[:, feat_idx["reactor_length_m"]]
        D_f   = X_ph[:, feat_idx["reactor_diameter_m"]]
        q_f   = X_ph[:, feat_idx["heat_flux_Wm2"]]
        m_f   = X_ph[:, feat_idx["mass_flow_rate_kgps"]].abs().clamp(min=1e-10)
        rhs   = L_f * torch.pi * D_f * q_f / (m_f * cp)
        T_ref = rhs.abs().detach().clamp(min=1.0)
        energy_per = ((dT_phys - rhs) / T_ref).pow(2).detach().cpu().numpy()

    return eos_per, mass_per, energy_per


# ── Trained PINN residuals ────────────────────────────────────────────────────
eos_pinn, mass_pinn, nrg_pinn = _eval_residuals(model, X_diag_colloc, z_L_diag_t)

# ── Untrained (random-weights) baseline ──────────────────────────────────────
torch.manual_seed(999)
baseline_model = PINNPFR(n_inputs, H1, H2, H3, n_outputs, dropout=0.0).to(device)
eos_base, mass_base, nrg_base = _eval_residuals(baseline_model, X_diag_colloc, z_L_diag_t)
del baseline_model

# ── Bin by z/L ────────────────────────────────────────────────────────────────
N_BINS = 20
bins   = np.linspace(0, 1, N_BINS + 1)
z_mid  = 0.5 * (bins[:-1] + bins[1:])


def _bin_mean(vals, z_arr):
    out = np.full(N_BINS, np.nan)
    for k in range(N_BINS):
        mask = (z_arr >= bins[k]) & (z_arr < bins[k + 1])
        if mask.sum() > 0:
            out[k] = float(np.mean(vals[mask]))
    return out


eos_pinn_p  = _bin_mean(eos_pinn,  z_L_diag)
mass_pinn_p = _bin_mean(mass_pinn, z_L_diag)
nrg_pinn_p  = _bin_mean(nrg_pinn,  z_L_diag)
eos_base_p  = _bin_mean(eos_base,  z_L_diag)
mass_base_p = _bin_mean(mass_base, z_L_diag)
nrg_base_p  = _bin_mean(nrg_base,  z_L_diag)

# ── Plot: PINN vs baseline per residual type ──────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
pairs = [
    (eos_pinn_p,  eos_base_p,  "EOS residual (normalised)"),
    (mass_pinn_p, mass_base_p, "Mass conservation residual"),
    (nrg_pinn_p,  nrg_base_p,  "Energy ODE residual"),
]
for ax, (pinn_p, base_p, label) in zip(axes, pairs):
    ax.plot(z_mid, base_p, 's--', color='0.6',  lw=1.5, ms=5, markerfacecolor='white', label='Untrained (random weights)')
    ax.plot(z_mid, pinn_p, 'o-',  color='b',    lw=2.0, ms=5, markerfacecolor='white', label='Trained PINN')
    ax.set_xlabel("z/L (relative position)")
    ax.set_ylabel("Mean squared residual")
    ax.set_title(label)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_xlim(0, 1)

plt.suptitle('Physics residual magnitude: PINN vs untrained baseline (test samples)', y=1.01)
plt.tight_layout()

if IF_PLOT_EXPORT:
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(FIG_DIR / 'physics_residual_along_z.png', dpi=150, bbox_inches='tight')
if IF_PLOT_SHOWN:
    plt.show()
plt.close(fig)

# ── Quantitative reduction table ─────────────────────────────────────────────
def _pct_reduction(pinn_arr, base_arr):
    p = np.nanmean(pinn_arr)
    b = np.nanmean(base_arr)
    if b == 0 or not np.isfinite(b):
        return float("nan")
    return 100.0 * (b - p) / b

print("\n" + "─" * 55)
print("Physics residual reduction: PINN vs random baseline")
print("─" * 55)
print(f"{'Constraint':<30} {'Baseline':>10} {'PINN':>10} {'Reduction':>12}")
print("─" * 55)
for name, pinn_arr, base_arr in [
    ("EOS",          eos_pinn,  eos_base),
    ("Mass conserv", mass_pinn, mass_base),
    ("Energy ODE",   nrg_pinn,  nrg_base),
]:
    b_mean = float(np.nanmean(base_arr))
    p_mean = float(np.nanmean(pinn_arr))
    red    = _pct_reduction(pinn_arr, base_arr)
    print(f"{name:<30} {b_mean:>10.4e} {p_mean:>10.4e} {red:>11.1f}%")
print("─" * 55)
print("Positive reduction = PINN enforces constraints better than random weights.")

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# 14. EXPORT MODEL ARTIFACTS
# ════════════════════════════════════════════════════════════════════════════
if IF_MODEL_EXPORT:
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    stem     = "pinn_pfr"
    run_at   = datetime.now().isoformat(timespec="seconds")

    state_path    = EXPORT_DIR / f"{stem}_state_dict.pt"
    scalers_path  = EXPORT_DIR / f"{stem}_scalers.joblib"
    manifest_path = EXPORT_DIR / f"{stem}_manifest.json"
    per_target_csv = EXPORT_DIR / f"{stem}_per_target_metrics.csv"

    torch.save(model.state_dict(), state_path)
    joblib.dump(
        {"scaler_X": scaler_X, "scaler_y": scaler_y, "label_encoder": label_encoder},
        scalers_path,
    )
    df_per_target.to_csv(per_target_csv, index=False)

    manifest = {
        "notebook":      "Main_8_PINN_PFR",
        "created_at":    run_at,
        "model_type":    "PINNPFR",
        "architecture": {
            "hidden_layers":  [H1, H2, H3],
            "dropout":        DROPOUT,
            "n_inputs":       n_inputs,
            "n_outputs":      n_outputs,
            "n_params":       n_params,
        },
        "training": {
            "epochs":                EPOCHS,
            "batch_size":            BATCH_SIZE,
            "learning_rate":         LEARNING_RATE,
            "early_stopped":         EARLY_STOPPED,
            "best_epoch":            best_epoch,
            "curriculum_warmup":     CURRICULUM_WARMUP,
            "lambda_data":           LAMBDA_DATA,
            "lambda_phys":           LAMBDA_PHYS,
            "n_colloc_per_batch":    N_COLLOC_PER_BATCH,
            "use_cantera_residuals": USE_CANTERA_RESIDUALS,
        },
        "data": {
            "source_pkl":      str(pkl_path.name),
            "split":           "run-level",
            "test_size":       TEST_SIZE,
            "n_train_rows":    int(len(X_train_df)),
            "n_test_rows":     int(len(X_test_df)),
            "n_train_runs":    int(len(train_runs)),
            "n_test_runs":     int(len(test_runs)),
            "feature_cols":    feature_cols,
            "target_cols":     target_cols,
            "state_targets":   state_target_cols,
            "species_targets": species_cols,
        },
        "metrics": {
            "test_r2":          float(test_r2),
            "test_r2_state":    float(test_r2_state),
            "test_r2_species":  float(test_r2_species),
            "train_r2":         float(train_r2),
            "train_r2_state":   float(train_r2_state),
            "train_r2_species": float(train_r2_species),
            "test_rmse":        float(test_rmse),
            "test_mae":         float(test_mae),
        },
        "files": {
            "state_dict":       str(state_path.name),
            "scalers":          str(scalers_path.name),
            "per_target_csv":   str(per_target_csv.name),
        },
    }

    with open(manifest_path, 'w') as f:
        json.dump(manifest, f, indent=2)

    print(f"[OK] state_dict:     {state_path}")
    print(f"[OK] scalers:        {scalers_path}")
    print(f"[OK] manifest:       {manifest_path}")
    print(f"[OK] per-target CSV: {per_target_csv}")
else:
    print("[SKIP] IF_MODEL_EXPORT=False — no files written.")


---

## Summary

- **Architecture**: dedicated `PINNPFR` class (`src/models/pinn.py`), 3 hidden layers, ReLU + Dropout. Physics residuals enforced externally via `torch.autograd.grad`.
- **Physics constraints** (active by default):
  - Algebraic: ideal-gas EOS, mass conservation `ρuA = ṁ`, species sum = 1, species ≥ 0.
  - ODE (autograd): energy balance `dT/d(z/L) = L π D q / (ṁ cp)` via `torch.autograd.grad`.
  - Cantera ODE residuals (`USE_CANTERA_RESIDUALS=True`): full `dY_k/dz` and `dT/dz` from production rates; requires raw (unlumped) species — see `src/physics/pfr_residuals.py`.
- **Curriculum**: `CURRICULUM_WARMUP_EPOCHS` of data-only training, then physics loss added with weight `LAMBDA_PHYS`.
- **Collocation**: `N_COLLOC_PER_BATCH` random z/L values per mini-batch — physics enforced at unseen axial positions without labels.
- **Diagnostics**: physics residual magnitude along z/L (§13) is the key PINN diagnostic. Low and uniform residuals indicate the model respects the governing equations across the reactor.
- **Comparison**: §11b compares test R² against Main_6 (exit only) and Main_7 (full profile, no physics).
